# ⚙️ Tokens y Embeddings

**Laboratorio de PLN — IFTS24**
Matías Barreto, 2026

**Encuentro 13 · Bloque 2 — 50 minutos**

---

## Objetivo

Entender cómo los LLMs convierten texto en números — y por qué eso determina qué pueden y no pueden hacer.

## Al terminar este bloque vas a poder:

1. Comparar cómo distintos tokenizadores descomponen texto en español.
2. Obtener embeddings vectoriales y calcular similitud semántica entre oraciones.
3. Implementar un buscador semántico simple con Sentence-BERT.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Token** | Unidad mínima que procesa un LLM: puede ser una palabra entera, una subpalabra o un carácter. |
| **Tokenización** | Proceso de dividir texto en tokens y convertirlos en IDs numéricos. |
| **Vocabulario** | El conjunto completo de tokens únicos que un modelo conoce y puede generar. |
| **Embedding** | Vector numérico que representa el significado de un token o una oración en un espacio geométrico. |
| **Similitud coseno** | Métrica que mide qué tan "parecidos" son dos vectores (1 = idénticos, 0 = sin relación). |

## ⚙️ Instalación de dependencias

In [ ]:
%%capture
# Actualizamos pip y limpiamos instalaciones previas
!pip install --upgrade pip
!pip uninstall -y transformers accelerate

# Instalamos las bibliotecas necesarias
!pip install transformers accelerate
!pip install sentence-transformers gensim scikit-learn numpy scipy

## Parte 1 — Tokenización

### Analogía

Un tokenizador es como el sílabo de un idioma: antes de entender una oración, la desarmás en sus unidades mínimas. Pero a diferencia de la silabificación humana, los tokenizadores no parten por sílabas sino por frecuencia: fragmentan menos las palabras comunes y más las palabras raras o en idiomas con menos representación en el corpus.

### Dónde vive esto en el mundo real

Cada vez que usás ChatGPT o Claude, tu texto pasa primero por un tokenizador. El costo de las APIs de LLMs se cobra **por token**. Tokenizar mal el español puede hacer que el modelo "vea" una palabra conocida como una secuencia de fragmentos desconocidos — y genere peores respuestas. Por eso existen modelos entrenados específicamente en español como Salamandra o BETO.

In [ ]:
from transformers import AutoTokenizer

# Función auxiliar para visualizar tokens con colores
colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def mostrar_tokens(texto, nombre_tokenizador):
    """
    Visualiza cómo un tokenizador divide el texto.

    Parámetros:
    - texto: String a tokenizar
    - nombre_tokenizador: Identificador del modelo en Hugging Face
    """
    print(f"\n{'='*60}")
    print(f"Tokenizador: {nombre_tokenizador}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(nombre_tokenizador)
    token_ids = tokenizer(texto).input_ids

    # Mostramos cada token con un color diferente
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )
    print(f"\n\nTotal de tokens: {len(token_ids)}")

### Texto de prueba

El texto combina varios desafíos:
- Español con mayúsculas y acentos
- Emoji (🧉) y carácter especial (ñ)
- Código Python y operadores
- Expresiones coloquiales argentinas

In [ ]:
# Texto de ejemplo con varios casos interesantes
texto = """
Español y MAYÚSCULAS
🧉 ñ
show_tokens False None elif == >= else: dos tabs:"    " Tres tabs: "       "
12.0*50=600
El mate está muy caliente
"""

print("Texto original:")
print(texto)

### Comparación de tokenizadores

Vamos a probar cuatro tokenizadores sobre el mismo texto:

| Tokenizador | Tipo | Característica |
|---|---|---|
| `bert-base-uncased` | WordPiece | Convierte todo a minúsculas |
| `bert-base-cased` | WordPiece | Mantiene mayúsculas |
| `Phi-3-mini-4k-instruct` | Tiktoken moderno | Optimizado para instrucciones |
| `gpt2` | BPE clásico | Mayormente entrenado en inglés |

In [ ]:
# BERT uncased (sin distinción de mayúsculas)
mostrar_tokens(texto, "bert-base-uncased")

In [ ]:
# BERT cased (con distinción de mayúsculas)
mostrar_tokens(texto, "bert-base-cased")

In [ ]:
# Phi-3 (modelo moderno)
mostrar_tokens(texto, "microsoft/Phi-3-mini-4k-instruct")

In [ ]:
# GPT-2 (tokenizador clásico)
mostrar_tokens(texto, "gpt2")

### ✎ Para pensar

- ¿Por qué el modelo entrenado principalmente en inglés fragmenta más las palabras en español?
- Si la API de un LLM cobra por token, ¿cómo impacta el tokenizador en el costo de usar el modelo?

## Parte 2 — Embeddings contextualizados

### Analogía

Un embedding es como la dirección de una palabra en un mapa de significados. Las palabras cercanas en el mapa tienen significados similares. La diferencia con un diccionario es que la "dirección" de una palabra cambia según el contexto: "banco" está en una zona distinta del mapa cuando aparece en "banco de datos" que en "sentarme en el banco".

### Dónde vive esto en el mundo real

Los embeddings son el corazón de los sistemas de búsqueda semántica de Notion, Google Docs y Slack. Son la tecnología que hace posible el RAG (Retrieval-Augmented Generation) que vamos a ver en el próximo encuentro. Sin embeddings, no hay forma de que un sistema "entienda" que "perro" y "can" significan lo mismo.

In [ ]:
from transformers import AutoModel, AutoTokenizer

# Cargamos un modelo BERT pequeño
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

print("Modelo DeBERTa cargado exitosamente")

### DeBERTa: modelo de embeddings contextualizados

Usamos `deberta-v3-xsmall` porque es liviano (22M parámetros) y rápido — suficiente para ver el concepto en acción.

Cada token del texto pasa por el modelo y sale como un **vector de 384 números** que captura su significado en ese contexto específico.

In [ ]:
# Procesamos una oración simple
oracion = 'Hola mundo'

# Tokenizamos
# return_tensors='pt': Devuelve tensores de PyTorch
tokens = tokenizer(oracion, return_tensors='pt')

print("Tokens generados:")
print(tokens)
print("\nIDs de tokens:")
print(tokens['input_ids'])
print("\nDecodificación de cada token:")
for token_id in tokens['input_ids'][0]:
    print(f"ID {token_id}: '{tokenizer.decode(token_id)}'")

In [ ]:
# Obtenemos los embeddings
# **tokens: Desempaqueta el diccionario como argumentos nombrados
# [0]: Tomamos la primera salida (hidden states)
output = model(**tokens)[0]

print("Forma del tensor de embeddings:")
print(output.shape)
print("\nInterpretación:")
print(f"- Batch size: {output.shape[0]} (cantidad de oraciones procesadas simultáneamente)")
print(f"- Sequence length: {output.shape[1]} (cantidad de tokens en la oración)")
print(f"- Hidden size: {output.shape[2]} (dimensiones del embedding de cada token)")

In [ ]:
# Visualizamos el embedding del primer token
import numpy as np

primer_token_embedding = output[0][0].detach().numpy()

print("Embedding del primer token [CLS]:")
print(f"Primeros 10 valores: {primer_token_embedding[:10]}")
print(f"\nEstadísticas del vector:")
print(f"- Media: {primer_token_embedding.mean():.4f}")
print(f"- Desviación estándar: {primer_token_embedding.std():.4f}")
print(f"- Mínimo: {primer_token_embedding.min():.4f}")
print(f"- Máximo: {primer_token_embedding.max():.4f}")

### ✎ Para pensar

- El tensor de embeddings tiene forma `[1, n_tokens, 384]`. ¿Qué representa cada una de esas tres dimensiones?
- Si dos tokens tienen embeddings muy similares, ¿qué nos dice eso sobre su significado?

## Parte 3 — Embeddings de oraciones y búsqueda semántica

### Analogía

Hasta ahora teníamos un vector por **token**. Pero para comparar documentos enteros necesitamos un vector por **oración**. Es como pasar de tener la coordenada de cada letra de una dirección a tener la coordenada del edificio completo.

### Dónde vive esto en el mundo real

Sentence-BERT es lo que usa un sistema de RAG para encontrar qué fragmento de un documento responde mejor tu pregunta. En este curso, en el próximo encuentro vas a usar exactamente esta tecnología para construir un sistema de preguntas y respuestas sobre tus propios documentos.

In [ ]:
from sentence_transformers import SentenceTransformer

# Cargamos un modelo de embeddings de oraciones
# all-mpnet-base-v2: Uno de los mejores modelos multilingües
modelo_oraciones = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

print("Modelo de sentence embeddings cargado")

In [ ]:
# Generamos embedding de una oración
oracion_ejemplo = "La mejor película que vi este año fue increíble"

# encode(): Convierte texto en embedding
vector = modelo_oraciones.encode(oracion_ejemplo)

print(f"Oración: '{oracion_ejemplo}'")
print(f"\nDimensiones del embedding: {vector.shape}")
print(f"Tipo de datos: {type(vector)}")
print(f"\nPrimeros 10 valores del vector:")
print(vector[:10])

### Búsqueda semántica: el concepto en acción

La búsqueda semántica no busca palabras exactas — busca **significado**. Una consulta sobre "comidas típicas" puede encontrar documentos que hablan de "mate" y "asado" aunque no compartan ninguna palabra con la consulta.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Definimos varias oraciones para comparar
oraciones = [
    "El perro corre por el parque",
    "Un can juega en el espacio verde",
    "El gato duerme en el sillón",
    "Hace mucho calor en verano",
    "La temperatura está muy alta en esta época"
]

# Generamos embeddings para todas las oraciones
embeddings = modelo_oraciones.encode(oraciones)

print(f"Generados {len(embeddings)} embeddings de {embeddings[0].shape[0]} dimensiones\n")

# Calculamos similitud entre la primera oración y las demás
print(f"Oración de referencia: '{oraciones[0]}'\n")
print("Similitud con otras oraciones:")
print("-" * 70)

referencia = embeddings[0].reshape(1, -1)

for i, oracion in enumerate(oraciones[1:], 1):
    similitud = cosine_similarity(referencia, embeddings[i].reshape(1, -1))[0][0]
    print(f"{similitud:.4f} | {oracion}")

In [ ]:
# Base de datos de documentos (simulada)
documentos = [
    "El mate es una bebida tradicional argentina que se prepara con yerba",
    "El asado argentino se caracteriza por su cocción lenta sobre brasas",
    "Buenos Aires es la capital de Argentina y su ciudad más poblada",
    "El tango es un género musical nacido en los suburbios de Buenos Aires",
    "La Patagonia argentina es conocida por sus paisajes montañosos y glaciares",
    "El dulce de leche es un producto típico de la repostería rioplatense",
    "El fútbol es el deporte más popular en Argentina",
    "La lengua oficial de Argentina es el español rioplatense"
]

# Generamos embeddings de todos los documentos
embeddings_documentos = modelo_oraciones.encode(documentos)
print(f"Base de datos indexada: {len(documentos)} documentos\n")

# Función de búsqueda
def buscar(consulta, top_k=3):
    """
    Busca los documentos más similares a la consulta.

    Parámetros:
    - consulta: Texto de búsqueda
    - top_k: Cantidad de resultados a devolver
    """
    # Generamos embedding de la consulta
    embedding_consulta = modelo_oraciones.encode([consulta])

    # Calculamos similitudes
    similitudes = cosine_similarity(embedding_consulta, embeddings_documentos)[0]

    # Obtenemos los índices de los top_k más similares
    # argsort(): Ordena y devuelve índices
    # [::-1]: Invertimos para orden descendente
    # [:top_k]: Tomamos los primeros k
    indices_top = similitudes.argsort()[::-1][:top_k]

    print(f"Consulta: '{consulta}'\n")
    print("Resultados:")
    print("=" * 80)
    for rank, idx in enumerate(indices_top, 1):
        print(f"\n{rank}. Similitud: {similitudes[idx]:.4f}")
        print(f"   {documentos[idx]}")

# Probamos varias consultas
buscar("¿Qué comidas típicas hay?", top_k=3)

In [ ]:
print("\n" + "="*80 + "\n")
buscar("Información sobre música y baile", top_k=3)

In [ ]:
print("\n" + "="*80 + "\n")
buscar("Lugares para visitar", top_k=3)

In [ ]:
# Agregá tus propios documentos
mis_documentos = [
    "Tu documento 1",
    "Tu documento 2",
    "Tu documento 3",
    # Agrega más...
]

# Indexá
mis_embeddings = modelo_oraciones.encode(mis_documentos)

# Buscá
mi_consulta = "Tu consulta aquí"
embedding_consulta = modelo_oraciones.encode([mi_consulta])
similitudes = cosine_similarity(embedding_consulta, mis_embeddings)[0]

# Mostrá resultados
for i, doc in enumerate(mis_documentos):
    print(f"{similitudes[i]:.4f} | {doc}")

### ✎ Para pensar

- La búsqueda semántica encontró "mate" y "asado" ante la consulta "comidas típicas". ¿Qué tiene que saber el modelo para hacer esa conexión?
- ¿Qué limitaciones tiene este buscador? ¿Qué pasaría si los documentos fueran en un idioma muy diferente al inglés?

In [ ]:
# ─── Espacio de práctica ──────────────────────────────────────────────────────
#
# Construí tu propio mini buscador semántico:
#   1. Armá una lista de 5-8 oraciones sobre un tema que te interese
#   2. Generá sus embeddings con modelo_oraciones.encode()
#   3. Buscá con al menos 3 consultas distintas
#   4. Observá: ¿el resultado tiene sentido semántico?
#
# Bonus: probá una consulta en inglés sobre documentos en español.
#        ¿El modelo multilingüe la maneja?
#
# ─────────────────────────────────────────────────────────────────────────────

mis_documentos = [
    "Oración 1",
    "Oración 2",
    "Oración 3",
]

mis_embeddings = modelo_oraciones.encode(mis_documentos)
mi_consulta = "Tu consulta aquí"
embedding_consulta = modelo_oraciones.encode([mi_consulta])

from sklearn.metrics.pairwise import cosine_similarity
similitudes = cosine_similarity(embedding_consulta, mis_embeddings)[0]

for doc, sim in zip(mis_documentos, similitudes):
    print(f"{sim:.4f} | {doc}")

## ⛰️ Cierre del bloque

| Concepto | Qué aprendiste |
|---|---|
| **Tokenización** | El texto se parte en tokens antes de entrar al modelo; el algoritmo varía por modelo |
| **Embeddings contextualizados** | Cada token → vector que cambia según el contexto de la oración |
| **Sentence-BERT** | Un vector por oración completa; permite comparar oraciones por significado |
| **Similitud coseno** | Métrica que mide la cercanía entre vectores (1 = idénticos) |
| **Búsqueda semántica** | Busca por significado, no por palabras exactas |

**Próximo bloque →** Aplicaciones de Transformers con pipelines de Hugging Face: vas a usar modelos listos para producción en análisis de sentimiento, clasificación y traducción.